# Adding Your Own Model to DeepBullwhip\n\nThis notebook shows the 3-step pattern for extending DeepBullwhip:\n1. Subclass the appropriate ABC\n2. Add the `@register` decorator\n3. Run the benchmark\n\nWe'll create a custom "panic ordering" policy that doubles orders when inventory drops below zero.

## Step 1: Define the Custom Policy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from deepbullwhip.policy.base import OrderingPolicy
from deepbullwhip.registry import register
from deepbullwhip.diagnostics.plots import COLORS, _apply_style, DOUBLE_COL, SINGLE_COL, GOLDEN

_apply_style()


@register("policy", "panic_ordering")
class PanicOrderingPolicy(OrderingPolicy):
    """Doubles the OUT order when inventory position is negative.

    This models a common real-world behavior where managers panic-order
    when they see stockouts, which amplifies the bullwhip effect.
    """

    def __init__(self, lead_time: int, service_level: float = 0.95,
                 panic_multiplier: float = 2.0):
        self.lead_time = lead_time
        self.service_level = service_level
        self.z_alpha = stats.norm.ppf(service_level)
        self.panic_multiplier = panic_multiplier

    def compute_order(self, inventory_position, forecast_mean, forecast_std):
        S = (self.lead_time + 1) * forecast_mean + (
            self.z_alpha * forecast_std * np.sqrt(self.lead_time + 1)
        )
        order = max(0.0, S - inventory_position)

        # Panic: double the order if inventory position is negative
        if inventory_position < 0:
            order *= self.panic_multiplier

        return order


print("PanicOrderingPolicy registered successfully!")

## Step 2: Verify Registration

In [ ]:
from deepbullwhip import list_registered

print("Registered policies:", list_registered("policy"))

## Step 3: Run the Benchmark\n\nNow we can use our custom policy alongside built-in policies.

In [ ]:
from deepbullwhip.benchmark import BenchmarkRunner

runner = BenchmarkRunner(
    chain_config="consumer_2tier",
    demand="semiconductor_ar1",
    T=156,
    N=20,
    seed=42,
)

results = runner.run(
    policies=[
        "order_up_to",
        ("proportional_out", {"alpha": 0.5}),
        "panic_ordering",  # Our custom policy!
    ],
    metrics=["BWR", "FILL_RATE", "TC"],
)

pivot = results.pivot_table(
    index=["policy", "echelon"],
    columns="metric",
    values="value",
    aggfunc="mean",
)
print(pivot.to_string(float_format="%.3f"))

## Results\n\nLet's visualize the BWR comparison to see how panic ordering amplifies the bullwhip.

In [ ]:
POLICY_LABELS = {"order_up_to": "OUT", "proportional_out": "POUT", "panic_ordering": "Panic"}
POLICY_COLORS = [COLORS["E1"], COLORS["E2"], "#c44040"]

e1 = results[results["echelon"] == "E1"]
e1_bwr = e1[e1["metric"] == "BWR"].copy().sort_values("value")
e1_fr = e1[e1["metric"] == "FILL_RATE"].copy()
# Align fill rate order to BWR order
e1_fr = e1_fr.set_index("policy").loc[e1_bwr["policy"].values].reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(DOUBLE_COL, DOUBLE_COL / GOLDEN / 1.8))

labels = [POLICY_LABELS.get(p, p) for p in e1_bwr["policy"]]
colors = [POLICY_COLORS[i] for i in range(len(labels))]

# BWR
bars1 = ax1.barh(labels, e1_bwr["value"].values, color=colors,
                  edgecolor="white", linewidth=0.5, height=0.5)
ax1.axvline(x=1.0, color=COLORS["grid"], linestyle="--", linewidth=0.5)
ax1.set_xlabel("Bullwhip Ratio (BWR)")
for bar, val in zip(bars1, e1_bwr["value"].values):
    ax1.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height() / 2,
             f"{val:.2f}", va="center", fontsize=7, color="#333333")

# Fill Rate
bars2 = ax2.barh(labels, e1_fr["value"].values, color=colors,
                  edgecolor="white", linewidth=0.5, height=0.5)
ax2.set_xlabel("Fill Rate")
for bar, val in zip(bars2, e1_fr["value"].values):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
             f"{val:.0%}", va="center", fontsize=7, color="#333333")

fig.savefig("custom_policy_comparison.pdf", dpi=300, bbox_inches="tight")
fig.savefig("custom_policy_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

As expected, the panic ordering policy amplifies the bullwhip effect significantly compared to OUT and POUT. This confirms that reactive, fear-based ordering behavior worsens supply chain dynamics.\n\nThe 3-step pattern (`subclass` + `@register` + `BenchmarkRunner`) makes it easy to test any custom policy hypothesis against established baselines.